# MCP(Model context Protocol)

### 用于便捷地将AI应用连接外部系统
### 分为McpServer和McpClient两部分
### 分别用于服务端和客户端,支持Stdio和Steamable HTTP两种协议

## 1.MCP通信示例

### · 基于Stdio通信

In [1]:
# 环境配置
import sys
import asyncio
from dotenv import load_dotenv
load_dotenv()

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient


# 注意配置transport参数
client = MultiServerMCPClient(
    {
          "time": {
            "transport" : "stdio",
            "command": "uvx",
            "args": ["mcp-server-time"]
          }
        }
)

tools = await client.get_tools()

In [3]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print("-" * 67)

get_current_time
Get current time in a specific timezone
-------------------------------------------------------------------
convert_time
Convert time between timezones
-------------------------------------------------------------------
get_time_ago
Get the time a given number of days, hours, minutes and seconds ago in a specific timezone
-------------------------------------------------------------------


In [5]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-flash",
    tools=tools
)

# 异步调用
response = await agent.ainvoke(
    {"messages" : [HumanMessage(content="现在几点了？")]}
)

for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

现在几点了？
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_Vzm9xrYbBGmFYuCoImSm1448)
 Call ID: call_00_Vzm9xrYbBGmFYuCoImSm1448
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-21T10:13:21+08:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_d21e78b0-12cc-4f86-881f-4c9a986b8c42'}]
================================== Ai Message ==================================

现在是 **2026年11月21日（星期六）10:13:21**，北京时间（Asia/Shanghai，UTC+8）。

如果你在其他时区，告诉我一声，我可以帮你换算。


### · 基于Streamable HTTP通信

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient


# 注意配置transport参数
client = MultiServerMCPClient(
    {
          "kiwi-com-filght-search": {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
          },
          "time": {
            "transport" : "stdio",
            "command": "uvx",
            "args": ["mcp-server-time"]
          }
        }
)

tools = await client.get_tools()

In [7]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print("-" * 67)

search-flight
# Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `tripType` — `"oneway"` or `"return"`
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO ti

In [9]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="你是一个旅游助手，帮助用户找到最好的航班，不要追问问题，如果用户使用中文，则使用zh-cn作为区域设置"
)

# 异步调用
response = await agent.ainvoke(
    {"messages" : [HumanMessage(content="帮我查询下周一从上海到大阪的航班")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查询下周一从上海到大阪的航班
================================== Ai Message ==================================

我先确认一下今天的日期，以便计算下周一的具体日期。
Tool Calls:
  get_current_time (call_00_l7EnbiHQZN7Qnhs462XG8426)
 Call ID: call_00_l7EnbiHQZN7Qnhs462XG8426
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-21T10:40:01+08:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_846fdad7-e9cb-4604-a1c1-2c5fa76dcf5f'}]
================================== Ai Message ==================================

今天是 2026年11月21日（周六），下周一为 **2026年11月23日**。我来为您搜索上海到大阪的航班。
Tool Calls:
  search-flight (call_00_gIFkjHsAJlEaqPNeRO8o7610)
 Call ID: call_00_gIFkjHsAJlEaqPNeRO8o7610
  Args:
    flyFrom: 上海
    flyTo: 大阪
    departureDate: 23/11/2026
    locale: zh-cn
   

## 2.自定义MCP客户端

In [ ]:
# 不要在notebook里面执行本代码块！！！

from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient


# 注意配置transport参数
client = MultiServerMCPClient(
    {
          "math": {
            "transport" : "stdio",
            "command" : "python",
            "args" : ["math.py"]
          }
        }
)

tools = await client.get_tools()

In [12]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print("-" * 67)

add
Add two numbers together
-------------------------------------------------------------------
multiply
Multiply two numbers together
-------------------------------------------------------------------


In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="你是一个数学小助手，你可以调用工具来帮你完成任务"
)

# 异步调用
response = await agent.ainvoke(
    {"messages" : [HumanMessage(content="帮我计算一下2的3次方与8的平方的和是多少")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我计算一下2的3次方与8的平方的和是多少
================================== Ai Message ==================================
Tool Calls:
  multiply (call_00_dSCSU9MjTcAsl7st9Q3w7641)
 Call ID: call_00_dSCSU9MjTcAsl7st9Q3w7641
  Args:
    a: 2
    b: 2
  multiply (call_01_QqZM5U7KeUAX17WW33gH8671)
 Call ID: call_01_QqZM5U7KeUAX17WW33gH8671
  Args:
    a: 8
    b: 8
================================= Tool Message =================================
Name: multiply

[{'type': 'text', 'text': '4.0', 'id': 'lc_6f2a2d39-6600-493b-921a-e44387b413f1'}]
================================= Tool Message =================================
Name: multiply

[{'type': 'text', 'text': '64.0', 'id': 'lc_ccc4cb9c-5d3a-4528-9f25-2f731c410eb6'}]
================================== Ai Message ==================================
Tool Calls:
  multiply (call_00_IRkoMsWZSncwyWppaFGQ7747)
 Call ID: call_00_IRkoMsWZSncwyWppaFGQ7747
  Args:
    a: 4
    b: 2
====